In [26]:


import os
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

data_folder = '../data/'
path = os.path.join(data_folder)

df = pd.DataFrame()
df2 = pd.DataFrame()
for file in os.listdir(path):
    if 'price_history_full' in file:
        df = pd.read_csv(data_folder + file)
        

launch_path = '../data/official_launch_prices.csv' #added on 26.05
launch_ref = pd.read_csv(launch_path)
launch_ref.head()

df = df.merge(launch_ref, on = 'submodel_name')
# meta_path = '../data/all_asins_meta.csv'
# meta_df = pd.read_csv(meta_path)

df.dtypes
df['official_premiere_date'] = pd.to_datetime(df['official_premiere_date'])
print(df[df['official_launch_price'].isna()]['submodel_name'].unique())



# KEEPA_EPOCH = pd.Timestamp('2011-01-01') #19.05 - - - Not needed anymore, we've already handled it before importing to SQL
# df['listed_since'] = pd.to_datetime(KEEPA_EPOCH + pd.to_timedelta(df['listed_since'], 'minutes'))

df['datetime'] = pd.to_datetime(df['datetime'])
df['days_since_launch'] = (df['datetime'] - df['official_premiere_date']).dt.days
df = df[df['days_since_launch'] > 0]

df['NEW'] = df['new_price'] * 100 #had to implement this here in the new pipeline, as it wasn't handled
df = df.drop('new_price', axis = 1)

df['price_pct_of_launch'] = round(df['NEW'] / df['official_launch_price'] * 100, 1)
df['days_rounded'] = (df['days_since_launch'] / 7).round() * 7


# df['median'] = df.groupby('asin')['NEW'].transform('median')
# df = df[df['NEW'] < 3 * df['median']]
# df = df[df['NEW'].notna()]

# df = df.set_index('datetime', drop = True)
# backup_df = df
# df = df.groupby('asin').resample('W')['NEW'].min().reset_index()
# df['NEW'] = df.groupby('asin')['NEW'].ffill()
# '''

# df = df.merge(meta_df[['asin', 'brand', 'submodel_name', 'generation_name', 'listed_since']], on='asin', how='left')
# df = df.rename(columns = {'generation_name': 'model'})

# df['listed_since'] = pd.to_datetime(df['listed_since'], utc = True)
# df['days_since_launch'] = (df['datetime'] - df['listed_since']).dt.days


# '''Computing first price, price_pct_of_launch etc'''

# df['first_price'] = df.groupby('submodel_name')['NEW'].transform('first')

# df['price_pct_of_launch'] = round(df['NEW'] / df['first_price'] * 100, 1)
# df['days_rounded'] = (df['days_since_launch'] / 7).round() * 7


<StringArray>
[]
Length: 0, dtype: str


In [44]:

#Created on 25.05, updated on 26.05 to account for THE ACTUAL OFFICIAL PREMIERE DATES (mean of all memory size prices) and launch prices for each submodel
ms_df = pd.read_csv('../data/monthly_sold_full.csv')

ms_df = ms_df.merge(launch_ref, on = 'submodel_name', how = 'left')
ms_df.head()

ms_df['official_premiere_date'] = pd.to_datetime(ms_df['official_premiere_date'])
ms_df['datetime'] = pd.to_datetime(ms_df['datetime'])

del(ms_df['premiere_date']) #deleting it so it doesn't confuse us later - the official premiere date is what matters

ms_df['days_since_launch'] = (ms_df['datetime'] - ms_df['official_premiere_date']).dt.days
ms_df['days_rounded'] = (ms_df['days_since_launch'] / 7).round() * 7
ms_df = ms_df[ms_df['days_since_launch'] >= 0]

ms_df['tier'] = ms_df.apply(
    lambda row: row['submodel_name'].replace(row['generation_name'], '').strip(),
    axis=1
)
ms_df['tier'] = ms_df['tier'].replace('', 'Base')

print(ms_df['tier'].value_counts())
print(ms_df.head())
print(ms_df.shape)


tier
Base        6228
Pro         4007
Pro Max     2309
Ultra       1577
a           1053
FE           965
Mini         781
Plus         737
+            730
Pro Fold      42
Edge          32
Pro XL        24
Name: count, dtype: int64
         asin    product_grade submodel_name  brand generation_name  \
0  B011SDYBZW  Renewed Premium     iPhone 11  Apple       iPhone 11   
1  B011SDYBZW  Renewed Premium     iPhone 11  Apple       iPhone 11   
2  B011SDYBZW  Renewed Premium     iPhone 11  Apple       iPhone 11   
3  B011SDYBZW  Renewed Premium     iPhone 11  Apple       iPhone 11   
4  B011SDYBZW  Renewed Premium     iPhone 11  Apple       iPhone 11   

             datetime  monthly_sold  official_launch_price  \
0 2023-10-20 07:04:00            50                    766   
1 2023-11-28 13:34:00           100                    766   
2 2024-01-01 08:42:00            50                    766   
3 2024-01-01 18:16:00           100                    766   
4 2024-01-02 14:56:00       

In [ ]:
#W4 D1 T3 //// W4 D2 T3 --- 

'''
Now that price_pct_of_launch is anchored to real launch prices, rebuild Chart 1:

1. Add `tier` column if not already on df
2. Filter to `tier == 'Base'`
3. Groupby `generation_name + days_rounded` → mean `price_pct_of_launch`
4. Plotly line: x = days_rounded, y = price_pct_of_launch, color = generation_name
5. Add hline at y=50 and y=100
6. Title: "Price Decay — Base models across generations"

Do the curves now start close to 100%? If not, note what's still off.'''



df['tier'] = df.apply(
    lambda row: row['submodel_name'].replace(row['generation_name'], '').strip(),
    axis=1
)
df['tier'] = df['tier'].replace('', 'Base')



base_df = df[(df['tier'] == 'Base') & (df['brand'] == 'Apple')] 
base_df.head()

grouped_base_df = base_df.groupby(['generation_name', 'days_rounded'])['price_pct_of_launch'].agg('mean').reset_index()
grouped_base_df.head()
line_plot = px.line(
    grouped_base_df,
    x = 'days_rounded',
    y = 'price_pct_of_launch',
    color = 'generation_name',
    title = 'Price decay - Base models across generations'
)
line_plot.add_hline(y = 50)
line_plot.show()

In [32]:
#W4 D1 T4
'''Pick one generation with multiple tiers (e.g. iPhone 13 — has Base, Mini, Pro, Pro Max).
Filter `df` to that generation_name.
Groupby `submodel_name + days_rounded` → mean `price_pct_of_launch`.
Plot Plotly line: x = days_rounded, y = price_pct_of_launch, color = submodel_name.
Add hline at y=50. Title: "Price Decay — [generation] by submodel".

Does the Pro Max hold value better than the base model? Does the Mini decay faster?'''

generation_df = df[df['generation_name'] == 'iPhone 13']
grouped_generation_decay_df = generation_df.groupby(['submodel_name', 'days_rounded'])['price_pct_of_launch'].agg('mean').reset_index()

grouped_generation_decay_chart = px.line(
    grouped_generation_decay_df,
    x = 'days_rounded',
    y = 'price_pct_of_launch',
    color = 'submodel_name',
    title = 'Price decay - Iphone 13 by submodels'
)
grouped_generation_decay_chart.add_hline(y = 50)
grouped_generation_decay_chart.show()

In [ ]:
df[df['model'] == 'iPhone 13'].groupby('submodel_name').agg(
    first_date=('datetime', 'first'),
    first_price=('NEW', 'first'),
    premiere_date=('listed_since', 'first')
).assign(days_gap=lambda x: (x['first_date'] - x['premiere_date']).dt.days)


In [ ]:
'''
Re-run Charts 1 & 2 with the corrected `first_price` (grouped by `submodel_name`).
Check:
- Do Base model curves start at or near 100%?
- Do Pro/Pro Max curves still start lower (expected — tracking gap), but at least
  normalize against their own first price now?
- Does Chart 2 (within iPhone 13) look more sensible than before?

Note the remaining tracking gap effect in a markdown cell — one sentence is enough.
We're acknowledging it, not fixing it.'''

